<a href="https://colab.research.google.com/github/JeffersonRodrigues9/Automacoes_com_python/blob/main/AWS_S3_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Baixando arquivos do S3 na AWS de forma automática com base nos documentos listados no TXT.

import boto3
from pathlib import Path
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

BUCKET_NOME = ""
ARQUIVO_LISTA = r".txt"
PASTA_DESTINO = r""

s3 = boto3.client(
    "s3",
    aws_access_key_id="",    aws_secret_access_key=""
)

def ler_lista(caminho_arquivo):
    with open(caminho_arquivo, "r", encoding="utf-8") as f:
        linhas = f.readlines()
        return [linha.strip() for linha in tqdm(linhas, desc="Lendo lista", unit="linha") if linha.strip()]


def expandir_prefixos(entrada_lista):
    todas_chaves = []
    paginator = s3.get_paginator("list_objects_v2")

    for item in tqdm(entrada_lista, desc="Expandindo pastas", unit="item"):
        ultimo_segmento = item.rstrip("/").split("/")[-1]
        e_prefixo = item.endswith("/") or ("." not in ultimo_segmento)

        if e_prefixo:
            prefixo = item if item.endswith("/") else item.rstrip("/") + "/"
            for page in paginator.paginate(Bucket=BUCKET_NOME, Prefix=prefixo):
                for obj in page.get("Contents", []):
                    key = obj["Key"]
                    if not key.endswith("/"):
                        todas_chaves.append(key)
        else:
            todas_chaves.append(item)

    vistos = set()
    resultado = []
    for k in todas_chaves:
        if k not in vistos:
            vistos.add(k)
            resultado.append(k)
    return resultado

def baixar_arquivo(s3_key):
    try:
        caminho_local = Path(PASTA_DESTINO) / s3_key
        caminho_local.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(BUCKET_NOME, s3_key, str(caminho_local))
        return f"{s3_key} - OK"
    except Exception as e:
        return f"{s3_key} - Erro: {e}"

def baixar_todos(caminhos_s3, num_processos):
    with Pool(num_processos) as pool:
        for resultado in tqdm(pool.imap_unordered(baixar_arquivo, caminhos_s3), total=len(caminhos_s3), desc="Baixando"):
            tqdm.write(resultado)


if __name__ == "__main__":
    entradas = ler_lista(ARQUIVO_LISTA)
    caminhos_s3 = expandir_prefixos(entradas)
    num_processos = 8
    baixar_todos(caminhos_s3, num_processos)